In [2]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


In [3]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [4]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [5]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'Hi! Welcome—it’s lovely to meet you! 😊 How can I help?'

In [8]:
ed = fetch_website_contents("https://www.theverge.com/")
print(ed)

The Verge

Skip to main content
The Verge
The Verge logo.
Tech
Reviews
Science
Entertainment
AI
Policy
Notifications
Notifications
Hamburger Navigation Button
Notifications
Notifications
Hamburger Navigation Button
Navigation Drawer
The Verge
The Verge logo.
Login
/
Sign Up
close
Close
Search
Light
System
Dark
Tech
Expand
Amazon
Apple
Facebook
Google
Microsoft
Samsung
Business
See all tech
Reviews
Expand
Smart Home Reviews
Phone Reviews
Tablet Reviews
Headphone Reviews
See all reviews
Science
Expand
Space
Energy
Environment
Health
See all science
Entertainment
Expand
TV Shows
Movies
Audio
See all entertainment
AI
Expand
OpenAI
Anthropic
See all AI
Policy
Expand
Antitrust
Politics
Law
Security
See all policy
Gadgets
Expand
Laptops
Phones
TVs
Headphones
Speakers
Wearables
See all gadgets
Verge Shopping
Expand
Buying Guides
Deals
Gift Guides
See all shopping
Gaming
Expand
Xbox
PlayStation
Nintendo
See all gaming
Streaming
Expand
Disney
HBO
Netflix
YouTube
Creators
See all streaming
Transp

In [9]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [10]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [11]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
response.choices[0].message.content

'4. Even arithmetic gets it right sometimes.'

In [12]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [13]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': "\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nThe Verge\n\nSkip to main content\nThe Verge\nThe Verge logo.\nTech\nReviews\nScience\nEntertainment\nAI\nPolicy\nNotifications\nNotifications\nHamburger Navigation Button\nNotifications\nNotifications\nHamburger Navigation Button\nNavigation Drawer\nThe Verge\nThe Verge logo.\nLogin\n/\nSign Up\nclose\nClose\nSearch\nLight\nSystem\nDark\nTech\nExpand\nAmazon\nApple\nFacebook\nGoogle\nMicrosoft\nSamsung\nBusiness\nSee all tech\nReviews\nExpand\nSmart Home Reviews\nPhone Reviews\nTablet Reviews\nHeadphone Reviews

In [14]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-6-luna",
        reasoning_effort = "none",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [15]:
summarize("https://www.theverge.com/")

'The Verge’s featured stories cover the mystery of relentless political texts, a homemade AI Tamagotchi, spooky YouTube picks, favorite keyboards, and rapper Vic Mensa’s reliance on Voice Memos. Election-season spam: somehow still harder to unsubscribe from than actual news.'

In [16]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://www.theverge.com/")

The Verge’s homepage highlights tech, culture, and gadget stories: why political texts keep finding people who never donated, an AI-powered Tamagotchi built before Meta got around to it, spooky YouTube picks, a tour of Verge staff keyboards, and Vic Mensa’s devotion to Voice Memos. The political-text story is the only clear news item here; the rest are features and recommendations.